In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import random
import os
os.makedirs("data", exist_ok=True)

# 1. CSV 로딩 (경로 중요)
customers = pd.read_csv("data/raw/customers.csv")
orders = pd.read_csv(
    "data/raw/orders.csv",
    parse_dates=["order_purchase_timestamp"]
)
order_items = pd.read_csv("data/raw/order_items.csv")

# -----------------------------
# 2. purchase 이벤트 생성
# -----------------------------
purchase_df = (
    order_items
    .merge(
        orders[['order_id','customer_id','order_purchase_timestamp']],
        on='order_id'
    )
    .merge(
        customers[['customer_id']],
        on='customer_id'
    )
)

purchase_df = purchase_df.rename(columns={
    'customer_id': 'user_id',
    'order_purchase_timestamp': 'event_time'
})

purchase_df['event_type'] = 'purchase'
purchase_df = purchase_df[['user_id','product_id','event_time','event_type']]

# -----------------------------
# 3. click 이벤트 생성
# -----------------------------
click_rows = []

for user_id, group in purchase_df.groupby('user_id'):
    products = group['product_id'].values
    times = group['event_time']  # Series 유지

    n_clicks = random.randint(3, 10)
    sampled_idx = np.random.choice(len(products), n_clicks, replace=True)

    for idx in sampled_idx:
        click_rows.append([
            user_id,
            products[idx],
            times.iloc[idx] - timedelta(minutes=random.randint(1, 60)),
            'click'
        ])

click_df = pd.DataFrame(
    click_rows,
    columns=['user_id','product_id','event_time','event_type']
)

# -----------------------------
# 4. 합치기 + 샘플링
# -----------------------------
events_df = pd.concat([purchase_df, click_df], ignore_index=True)
events_df = events_df.sort_values(['user_id','event_time'])

events_df.sample(
    n=min(10000, len(events_df)),
    random_state=42
).to_csv("data/ecommerce_logs.csv", index=False)
print("로그 생성 완료")


로그 생성 완료
